# 卡车装载问题

**类别：** 装箱

来源: [https://www.hexaly.com/templates/truck-loading-problem](https://www.hexaly.com/templates/truck-loading-problem)


## 问题

在**卡车装载问题**中，给定的一组物品需要装载到卡车中。因此，每件物品必须恰好被分配到一辆卡车。

每辆卡车有两层：底层和上层。每一层具有相同的物品位置数量，每辆卡车也具有相同的载重能力。因此，分配给某辆卡车的物品总重量不得超过该载重能力。

除重量外，物品还可能存在堆放限制：

- 类型 1 物品必须放置在底层，并且其上方不能再放置其他物品，称为 “Alone（独占）” 物品。
- 类型 2 物品必须放置在底层，但其上方可以再放置其他物品，称为 “Floor（底层）” 物品。
- 类型 3 物品没有任何堆放限制，称为 “No-restriction（无限制）” 物品。
- 类型 4 物品上方不能再放置其他物品，但可以放在底层或上层，称为 “Delicate（易碎）” 物品。

此外，卡车的上层只有在底层被装满时才能使用。目标是最小化所使用的卡车数量。

	

### 学到的建模原则

- 添加 [set 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模各箱子中所装物品
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算卡车的总重量并强制执行堆放限制
- 在后处理函数中使用 Hexaly Optimizer 找到的解


## 数据

所提供的卡车装载问题实例改编自 [BPPLIB](https://site.unibo.it/operations-research/en/research/bpplib-a-bin-packing-problem-library) 中的 Falkenauer 实例。数据文件的格式如下：

- 第一行包含一个整数：物品数量。
- 第二行包含两个整数：每辆卡车的载重能力以及卡车每层的物品位置数。
- 之后每行使用两个整数描述一件物品：
- 首先是其重量，

- 然后是一个介于 1 到 4（含）之间的整数，指定其类型：
- 1 表示 “Alone（独占）” 物品，

- 2 表示 “Floor（底层）” 物品，

- 3 表示 “No-restriction（无限制）” 物品，

- 或 4 表示 “Delicate（易碎）” 物品。


## 模型

卡车装载问题的 Hexaly 模型使用 [set 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。对每辆卡车，我们定义一个表示分配给它的物品集合的 set 变量。我们约束这些 set 变量构成一个 [partition（划分）](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#partition)，以确保每件物品恰好属于一辆卡车。

我们使用对集合的可变参数 **sum（求和）** 运算符以及一个返回指定物品索引对应重量的 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算每辆卡车的总重量。需要注意的是，该求和的项数在搜索过程中会随着集合大小变化而变化。然后我们可以约束总重量不超过卡车的载重能力。

“Alone” 和 “Floor” 物品必须位于其所在卡车的底层。因此，模型保证在任何卡车中它们都不超过每层的可用位置数。接着我们对 “Alone” 和 “Delicate” 物品添加类似的约束，因为它们上方不能再放置其他物品。

“Alone” 物品必须放置在底层，并且由于其上方不能再放置物品，因此会同时占用对应的上方位置。也就是说，它们占用两个位置单位而非一个。然后模型保证每辆卡车中被占用的位置数不超过总位置数。

我们使用 count（计数）运算符计算使用的卡车总数，该运算符返回集合中元素的数量。

该模型仅给出每辆卡车所装载的物品集合。物品在卡车内的具体摆放则由一个后处理函数另行计算。该函数先装载 “Alone” 和 “Floor” 物品以确保它们位于底层，然后装载 “No-restriction” 物品，最后装载必然位于最上层的 “Delicate” 物品。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys


def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]

#
# Auxiliary function used in the later "insideTruck" function. 
# Inserts an item into the next free slot in the truck.
#
def place_first_available_pos(positions, item, level_spots_count):
    if len(positions[0]) < level_spots_count:
        positions[0].append(item)
    else:
        positions[1].append(item)

#
# The optimizer assigns items to trucks but does not position them inside each truck.  
# This function computes a feasible two-level placement for the items in a truck.
#
def inside_truck(truck_set, category_items, level_spots_count):
    # Split the items assigned to the truck by category
    set_a = []  # Type 1: floor level and nothing above. Called "Alone" items
    set_f = []  # Type 2: floor level. Called "Floor" items
    set_n = []  # Type 3: no restriction. Called "No-restriction" items
    set_d = []  # Type 4: nothing above. Called "Delicate" items

    for i in truck_set:
        category = category_items[i]
        if category == 1:
            set_a.append(i)
        elif category == 2:
            set_f.append(i)
        elif category == 3:
            set_n.append(i)
        else:
            set_d.append(i)

    # Initialize the two-level position list: positions[0] represents the floor level, and positions[1] the upper level
    positions = [[], []]

    #
    # Assign positions to items, from the most constraining category to the least constraining one: A -> F -> N -> D
    #

    # Place all "Alone" items on the floor, and leave the corresponding upper positions empty (represented by -1)
    for i in set_a:
        positions[0].append(i)
        positions[1].append(-1)

    # Place all "Floor" items on the floor
    for i in set_f:
        positions[0].append(i)

    # Place all "No-restriction" items at the first available position
    for i in set_n:
        place_first_available_pos(positions, i, level_spots_count)

    # Place all "Delicate" items at the first available position
    for i in set_d:
        place_first_available_pos(positions, i, level_spots_count)

    return positions

# Auxiliary function used to display the solution
def pad_left(x, width):
    return (" " if x == -1 else str(x)).rjust(width)


def solve(instance_file, sol_file, time_limit=10):
    #
    # Read instance data
    #
    file_it = iter(read_integers(instance_file))

    nb_items = int(next(file_it))
    truck_weight_capacity = int(next(file_it))
    level_spots_count = int(next(file_it))

    item_weights = []
    category_items = []
    volumes_data = []

    for _ in range(nb_items):
        weight = int(next(file_it))
        category = int(next(file_it))
        item_weights.append(weight)
        category_items.append(category)

        # "Alone" items block both a floor position and the upper position above it
        volumes_data.append(2 if category == 1 else 1)

    # Bounds on the number of used trucks
    nb_min_trucks = (sum(item_weights) + truck_weight_capacity - 1) // truck_weight_capacity
    nb_max_trucks = nb_items

    #
    # Declare the optimization model
    #
    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        model = optimizer.model

        # Set decisions: trucks[k] represents the items assigned to truck k
        trucks = [model.set(nb_items) for _ in range(nb_max_trucks)]

        # Each item must be assigned to exactly one truck
        model.constraint(model.partition(trucks))

        #
        # Create arrays and functions to retrieve the item's data
        #
        weights = model.array(item_weights)
        categories = model.array(category_items)
        volumes = model.array(volumes_data)

        weight_lambda = model.lambda_function(lambda i: weights[i])
        volume_lambda = model.lambda_function(lambda i: volumes[i])
        floor_lambda = model.lambda_function(
            lambda i: model.or_(categories[i] == 1, categories[i] == 2)
        )
        delicate_lambda = model.lambda_function(
            lambda i: model.or_(categories[i] == 1, categories[i] == 4)
        )

        truck_weights = []
        is_used = []

        for k in range(nb_max_trucks):
            truck_weight = model.sum(trucks[k], weight_lambda)
            truck_weights.append(truck_weight)

            # Weight constraint for each truck
            model.constraint(truck_weight <= truck_weight_capacity)

            # Volume constraint for each truck
            model.constraint(model.sum(trucks[k], volume_lambda) <= 2 * level_spots_count)

            # Type 1 and type 2 items must be placed on the floor level
            model.constraint(model.sum(trucks[k], floor_lambda) <= level_spots_count)

            # Type 1 and type 4 items cannot have another item on top of them
            model.constraint(model.sum(trucks[k], delicate_lambda) <= level_spots_count)

            # Truck k is used if at least one item is in it
            is_used.append(model.count(trucks[k]) > 0)

        # Count the used trucks
        total_used_trucks = model.sum(is_used)

        # Minimize the number of used trucks
        model.minimize(total_used_trucks)
        model.close()

        # Parametrize the optimizer
        optimizer.param.time_limit = time_limit

        # Stop the search if the lower threshold is reached
        optimizer.param.set_objective_threshold(0, nb_min_trucks)

        optimizer.solve()

        #
        # Write the solution in a file
        #
        if sol_file is not None:
            print_width = len(str(nb_items - 1)) + 2

            with open(sol_file, "w") as f:
                f.write(f"Number of used trucks: {total_used_trucks.value}\n\n")

                for k in range(nb_max_trucks):
                    if not is_used[k].value:
                        continue

                    truck = inside_truck(trucks[k].value, category_items, level_spots_count)
                    line = "-" * (level_spots_count * print_width)

                    f.write(f"Truck weight: {truck_weights[k].value}\n")
                    f.write(line + "\n")

                    for item in truck[1]:
                        f.write(pad_left(item, print_width))
                    f.write("\n")
                    for item in truck[0]:
                        f.write(pad_left(item, print_width))

                    f.write("\n")
                    f.write(line + "\n\n")


def main():
    if len(sys.argv) < 2:
        print("Usage: python truck_loading.py inputFile [outputFile] [timeLimit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    if (len(sys.argv) >= 4):
        time_limit = int(sys.argv[3])  
    else: 
        time_limit = 10

    if (len(sys.argv) >= 3):
        solve(instance_file, sys.argv[2], time_limit)
    else:
        solve(instance_file, None, time_limit)


if __name__ == "__main__":
    main()
